# End-to-End Inspect Damage Walkthrough (Server-Aligned)

This notebook shows how the server computes Inspect Damage values for one event file:

- `../data/raw/13998/v35_data_processing/4e1_bt1xx_bt1tc_vpi_701_20240828_lca_lf.csv`

It mirrors the production flow while **skipping database retrieval** (CSV is used as the source of channel series):

1. Parse RSP-style CSV (`#TITLES`, `#UNITS`, `#DATA`)
2. Apply v35 plot channel map (`x_col`, `y_col`)
3. Derive the 12 canonical Inspect Damage channels
4. Extract channel time series values
5. Run fatigue damage (`py_fatigue`) through `FatigueDamageCalculator`

Server reference modules:

- `../server/server/services/etl/csv_parser.py`
- `../server/server/services/ingestion.py` (`_channel_map_with_headers` behavior mirrored)
- `../server/server/services/damage_channels.py`
- `../server/server/services/fatigue_damage.py`
- `../server/server/services/query.py` (`get_damage_channel_series`, DB step skipped here)
- `../server/server/routers/damage.py`

In [2]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import py_fatigue as pf
import matplotlib.pyplot as plt

# Add Dashboard root to sys.path so `import server...` resolves consistently
cwd = Path.cwd().resolve()
search_roots = [cwd, *cwd.parents]
dashboard_root = None
for root in search_roots:
    if (root / "server" / "services" / "fatigue_damage.py").exists():
        dashboard_root = root
        break

if dashboard_root is None:
    raise RuntimeError("Could not locate Dashboard root containing server/services/fatigue_damage.py")

if str(dashboard_root) not in sys.path:
    sys.path.insert(0, str(dashboard_root))

from server.services.etl.csv_parser import CSVParser
from server.services.damage_channels import derive_damage_channel_specs
from server.services.fatigue_damage import FatigueDamageCalculator, ChannelSeries

pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 160)

In [3]:
event_csv_path = Path("../data/raw/13998/v35_data_processing/4e1_bt1xx_bt1tc_vpi_701_20240828_lca_lf.csv")
assert event_csv_path.exists(), f"CSV not found: {event_csv_path}"

parser = CSVParser()
parsed = parser.parse(event_csv_path.read_bytes(), event_csv_path.name)
assert parsed.is_valid, parsed.error

df = parsed.dataframe
headers = list(df.columns)
units = list(df.attrs.get("units", []))

print(f"Rows: {len(df):,}, Columns: {len(df.columns)}")
print("First 10 headers:")
for i, h in enumerate(headers[:10]):
    print(f"  {i}: {h}")

print("\nFirst 6 unit entries:")
for i, u in enumerate(units[:6]):
    print(f"  {i}: {u}")

df.head()

Rows: 6,144, Columns: 23
First 10 headers:
  0: 
  1: 
  2: 001_1 LF LCA OtrBJ P_UG_X Force
  3: 002_2 LF LCA OtrBJ P_UG_Y Force
  4: 003_3 LF LCA OtrBJ P_UG_Z Force
  5: 004_4 LF ShockLwBsh P_UG_X Force
  6: 005_5 LF ShockLwBsh P_UG_Y Force
  7: 006_6 LF ShockLwBsh P_UG_Z Force
  8: 007_7 LF LCABushingF P_UG_X Force
  9: 008_8 LF LCABushingF P_UG_Y Force

First 6 unit entries:
  0: 
  1: 
  2: N
  3: N
  4: N
  5: N


,,,001_1 LF LCA OtrBJ P_UG_X Force,002_2 LF LCA OtrBJ P_UG_Y Force,003_3 LF LCA OtrBJ P_UG_Z Force,004_4 LF ShockLwBsh P_UG_X Force,005_5 LF ShockLwBsh P_UG_Y Force,006_6 LF ShockLwBsh P_UG_Z Force,007_7 LF LCABushingF P_UG_X Force,008_8 LF LCABushingF P_UG_Y Force,...,012_12 LF LCABushingR P_UG_Z Force,013_13 LF ShockLwBsh P_UG_X Momt,014_14 LF ShockLwBsh P_UG_Y Momt,015_15 LF ShockLwBsh P_UG_Z Momt,016_16 LF LCABushingF P_UG_X Momt,017_17 LF LCABushingF P_UG_Y Momt,018_18 LF LCABushingF P_UG_Z Momt,019_19 LF LCABushingR P_UG_X Momt,020_20 LF LCABushingR P_UG_Y Momt,021_21 LF LCABushingR P_UG_Z Momt
0,1.0,0.000,54.54,-1151.0,10130.0,65.64,-4151.0,-13720.0,-126.8,1762.0,...,2733.0,30540.0,-1935.0,-213.9,6763.0,-1094.0,1894.0,9100.0,4728.0,-1692.0
1,2.0,0.001,53.57,-1170.0,10130.0,66.43,-4149.0,-13720.0,-127.3,1769.0,...,2734.0,30780.0,-1912.0,-222.5,6829.0,-1093.0,1903.0,9183.0,4727.0,-1702.0
2,3.0,0.002,53.18,-1190.0,10130.0,67.09,-4148.0,-13730.0,-127.7,1776.0,...,2735.0,31020.0,-1889.0,-232.7,6895.0,-1091.0,1910.0,9268.0,4725.0,-1712.0
3,4.0,0.003,53.28,-1208.0,10140.0,67.52,-4146.0,-13730.0,-128.4,1783.0,...,2735.0,31270.0,-1870.0,-242.7,6960.0,-1089.0,1918.0,9350.0,4723.0,-1723.0
4,5.0,0.004,54.15,-1227.0,10140.0,67.66,-4144.0,-13730.0,-129.3,1790.0,...,2736.0,31500.0,-1859.0,-251.8,7023.0,-1088.0,1925.0,9432.0,4719.0,-1735.0


In [4]:
# Standard v35 plot map (column-index based)
v35_plot_map = {
    "bj_xy_force_plot": {"x_col": 2, "y_col": 3},
    "bj_xz_force_plot": {"x_col": 2, "y_col": 4},
    "shock_xy_force_plot": {"x_col": 20, "y_col": 21},
    "shock_xz_force_plot": {"x_col": 20, "y_col": 22},
    "bushing_f_xy_force_plot": {"x_col": 8, "y_col": 9},
    "bushing_f_xz_force_plot": {"x_col": 8, "y_col": 10},
    "bushing_r_xy_force_plot": {"x_col": 14, "y_col": 15},
    "bushing_r_xz_force_plot": {"x_col": 14, "y_col": 16},
}


def map_with_headers(plot_map: dict, headers_list: list[str], units_list: list[str]) -> dict:
    """Mirror ingestion._channel_map_with_headers behavior for notebook use."""
    resolved = {}
    for plot_key, mapping in plot_map.items():
        x_col = int(mapping["x_col"])
        y_col = int(mapping["y_col"])
        resolved[plot_key] = {
            "x_col": x_col,
            "y_col": y_col,
            "x_channel": headers_list[x_col] if x_col < len(headers_list) else None,
            "y_channel": headers_list[y_col] if y_col < len(headers_list) else None,
            "x_unit": units_list[x_col] if x_col < len(units_list) else None,
            "y_unit": units_list[y_col] if y_col < len(units_list) else None,
        }
    return resolved


resolved_plot_map = map_with_headers(v35_plot_map, headers, units)

channel_map_rows = []
for plot_key, mapping in resolved_plot_map.items():
    channel_map_rows.append(
        {
            "plot_key": plot_key,
            "x_col": mapping["x_col"],
            "y_col": mapping["y_col"],
            "x_channel": mapping["x_channel"],
            "y_channel": mapping["y_channel"],
            "x_unit": mapping["x_unit"],
            "y_unit": mapping["y_unit"],
        }
    )

pd.DataFrame(channel_map_rows).sort_values("plot_key").reset_index(drop=True)

,plot_key,x_col,y_col,x_channel,y_channel,x_unit,y_unit
0,bj_xy_force_plot,2,3,001_1 LF LCA OtrBJ P_UG_X Force,002_2 LF LCA OtrBJ P_UG_Y Force,N,N
1,bj_xz_force_plot,2,4,001_1 LF LCA OtrBJ P_UG_X Force,003_3 LF LCA OtrBJ P_UG_Z Force,N,N
2,bushing_f_xy_force_plot,8,9,007_7 LF LCABushingF P_UG_X Force,008_8 LF LCABushingF P_UG_Y Force,N,N
3,bushing_f_xz_force_plot,8,10,007_7 LF LCABushingF P_UG_X Force,009_9 LF LCABushingF P_UG_Z Force,N,N
4,bushing_r_xy_force_plot,14,15,013_13 LF ShockLwBsh P_UG_X Momt,014_14 LF ShockLwBsh P_UG_Y Momt,Nmm,Nmm
5,bushing_r_xz_force_plot,14,16,013_13 LF ShockLwBsh P_UG_X Momt,015_15 LF ShockLwBsh P_UG_Z Momt,Nmm,Nmm
6,shock_xy_force_plot,20,21,019_19 LF LCABushingR P_UG_X Momt,020_20 LF LCABushingR P_UG_Y Momt,Nmm,Nmm
7,shock_xz_force_plot,20,22,019_19 LF LCABushingR P_UG_X Momt,021_21 LF LCABushingR P_UG_Z Momt,Nmm,Nmm


In [5]:
damage_specs = derive_damage_channel_specs(channel_map_rows)

spec_df = pd.DataFrame(
    [
        {
            "channel_key": spec.key,
            "label": spec.label,
            "plot_key": spec.plot_key,
            "axis": spec.axis,
            "channel_name": spec.channel_name,
            "unit": spec.unit,
            "error": spec.error,
        }
        for spec in damage_specs
    ]
)

print(f"Derived specs: {len(damage_specs)}")
spec_df

Derived specs: 12


,channel_key,label,plot_key,axis,channel_name,unit,error
0,bj_x_force,BJ X Force,bj_xy_force_plot,x,001_1 LF LCA OtrBJ P_UG_X Force,N,None
1,bj_y_force,BJ Y Force,bj_xy_force_plot,y,002_2 LF LCA OtrBJ P_UG_Y Force,N,None
2,bj_z_force,BJ Z Force,bj_xz_force_plot,y,003_3 LF LCA OtrBJ P_UG_Z Force,N,None
3,shock_x_force,Shock X Force,shock_xy_force_plot,x,019_19 LF LCABushingR P_UG_X Momt,Nmm,None
4,shock_y_force,Shock Y Force,shock_xy_force_plot,y,020_20 LF LCABushingR P_UG_Y Momt,Nmm,None
5,shock_z_force,Shock Z Force,shock_xz_force_plot,y,021_21 LF LCABushingR P_UG_Z Momt,Nmm,None
6,bushing_f_x_momt,Bushing F X Momt,bushing_f_xy_force_plot,x,007_7 LF LCABushingF P_UG_X Force,N,None
7,bushing_f_y_momt,Bushing F Y Momt,bushing_f_xy_force_plot,y,008_8 LF LCABushingF P_UG_Y Force,N,None
8,bushing_f_z_momt,Bushing F Z Momt,bushing_f_xz_force_plot,y,009_9 LF LCABushingF P_UG_Z Force,N,None
9,bushing_r_x_momt,Bushing R X Momt,bushing_r_xy_force_plot,x,013_13 LF ShockLwBsh P_UG_X Momt,Nmm,None


## Step A: Deep Dive on BJ X Force

This section traces one channel exactly like server logic:

- Resolve canonical damage channel (`bj_x_force`) from `derive_damage_channel_specs`
- Extract that channel's time-series values from parsed CSV
- Build cycle count (`CycleCount.from_timeseries`)
- Compute Palmgren-Miner damage through `FatigueDamageCalculator`

For this event, `bj_x_force` maps to raw channel:

- `001_1 LF LCA OtrBJ P_UG_X Force`

In [6]:
bj_spec = next(spec for spec in damage_specs if spec.key == "bj_x_force")
assert bj_spec.channel_name is not None, "BJ X spec channel_name is missing"

bj_raw = pd.to_numeric(df[bj_spec.channel_name], errors="coerce")
bj_values = bj_raw.dropna().to_numpy(dtype=float)

print("BJ spec:")
print({
    "channel_key": bj_spec.key,
    "label": bj_spec.label,
    "channel_name": bj_spec.channel_name,
    "unit": bj_spec.unit,
    "num_samples": int(bj_values.size),
})

plt.figure(figsize=(12, 4))
plt.plot(bj_values)
plt.title(f"{bj_spec.label} signal: {bj_spec.channel_name}")
plt.xlabel("Sample index")
plt.ylabel(f"Value ({bj_spec.unit or 'raw unit'})")
plt.grid(True, alpha=0.3)
plt.show()

BJ spec:
{'channel_key': 'bj_x_force', 'label': 'BJ X Force', 'channel_name': '001_1 LF LCA OtrBJ P_UG_X Force', 'unit': 'N', 'num_samples': 6144}


FigureCanvasAgg is non-interactive, and thus cannot be shown


In [7]:
# Match server FatigueDamageCalculator timing behavior: time = np.arange(n)
bj_time = np.arange(bj_values.size, dtype=float)

bj_cycle_count = pf.CycleCount.from_timeseries(
    bj_values,
    time=bj_time,
    mean_bin_width=100.0,
    range_bin_width=100.0,
)

bj_cycle_count

,None
Cycle counting object,
"largest full stress range, MPa",1044.77
"largest stress range, MPa",4632.0
number of full cycles,115
number of residuals,10
number of small cycles,0
stress concentration factor,N/A
residuals resolved,False
mean stress-corrected,No


In [8]:
calculator = FatigueDamageCalculator()

bj_server_result = calculator.calculate_channel(
    ChannelSeries(
        channel_key=bj_spec.key,
        channel_name=bj_spec.channel_name,
        unit=bj_spec.unit,
        values=bj_values,
    )
)

# Transparent raw py_fatigue equivalent using same constants as server
sn_curve = pf.SNCurve(
    [3, 5],
    intercept=[12.592, 16.320],
    norm="DNVGL-RP-C203/2016",
    environment="Air",
    curve="C",
)
py_damage = float(sum(pf.damage.stress_life.get_pm(cycle_count=bj_cycle_count, sn_curve=sn_curve)))

bj_compare_df = pd.DataFrame(
    [
        {
            "method": "server_calculator",
            "damage": bj_server_result.damage,
            "status": bj_server_result.status,
            "error": bj_server_result.error,
        },
        {
            "method": "raw_py_fatigue",
            "damage": py_damage,
            "status": "ok",
            "error": None,
        },
    ]
)

bj_compare_df

,method,damage,status,error
0,server_calculator,0.019451,ok,None
1,raw_py_fatigue,0.019451,ok,None


## Step B: Run all 12 Inspect Damage channels

This reproduces server-style per-channel damage output:

- `channel_key`
- `label`
- `unit`
- `damage`
- `status`
- `error`

In [9]:
summary_rows = []

for spec in damage_specs:
    if spec.error is not None or not spec.channel_name:
        summary_rows.append(
            {
                "channel_key": spec.key,
                "label": spec.label,
                "channel_name": spec.channel_name,
                "unit": spec.unit,
                "damage": None,
                "status": "unavailable",
                "error": spec.error or "Channel mapping unavailable",
            }
        )
        continue

    series_values = pd.to_numeric(df[spec.channel_name], errors="coerce").dropna().to_numpy(dtype=float)

    result = calculator.calculate_channel(
        ChannelSeries(
            channel_key=spec.key,
            channel_name=spec.channel_name,
            unit=spec.unit,
            values=series_values,
        )
    )

    summary_rows.append(
        {
            "channel_key": spec.key,
            "label": spec.label,
            "channel_name": spec.channel_name,
            "unit": spec.unit,
            "damage": result.damage,
            "status": result.status,
            "error": result.error,
            "num_samples": int(series_values.size),
        }
    )

summary_df = pd.DataFrame(summary_rows)
summary_df

,channel_key,label,channel_name,unit,damage,status,error,num_samples
0,bj_x_force,BJ X Force,001_1 LF LCA OtrBJ P_UG_X Force,N,0.019451,ok,None,6144
1,bj_y_force,BJ Y Force,002_2 LF LCA OtrBJ P_UG_Y Force,N,0.626617,ok,None,6144
2,bj_z_force,BJ Z Force,003_3 LF LCA OtrBJ P_UG_Z Force,N,0.260240,ok,None,6144
3,shock_x_force,Shock X Force,019_19 LF LCABushingR P_UG_X Momt,Nmm,137.818109,ok,None,6144
4,shock_y_force,Shock Y Force,020_20 LF LCABushingR P_UG_Y Momt,Nmm,0.008411,ok,None,6144
5,shock_z_force,Shock Z Force,021_21 LF LCABushingR P_UG_Z Momt,Nmm,0.286030,ok,None,6144
6,bushing_f_x_momt,Bushing F X Momt,007_7 LF LCABushingF P_UG_X Force,N,0.000821,ok,None,6144
7,bushing_f_y_momt,Bushing F Y Momt,008_8 LF LCABushingF P_UG_Y Force,N,0.031511,ok,None,6144
8,bushing_f_z_momt,Bushing F Z Momt,009_9 LF LCABushingF P_UG_Z Force,N,0.000061,ok,None,6144
9,bushing_r_x_momt,Bushing R X Momt,013_13 LF ShockLwBsh P_UG_X Momt,Nmm,3718.610253,ok,None,6144


In [10]:
# Sanity checks
assert len(summary_df) == 12, f"Expected 12 damage channels, got {len(summary_df)}"

bj_summary_damage = summary_df.loc[summary_df["channel_key"] == "bj_x_force", "damage"].iloc[0]
assert np.isfinite(bj_summary_damage), "BJ X damage is not finite"

assert np.isclose(
    bj_summary_damage,
    bj_server_result.damage,
    rtol=1e-12,
    atol=0.0,
), "BJ X deep-dive damage does not match summary table"

print("All sanity checks passed.")
print(f"BJ X damage: {bj_server_result.damage:.12g}")
print("All 12 channel statuses:")
print(summary_df[["channel_key", "status"]])

All sanity checks passed.
BJ X damage: 0.019450594645
All 12 channel statuses:
         channel_key status
0         bj_x_force     ok
1         bj_y_force     ok
2         bj_z_force     ok
3      shock_x_force     ok
4      shock_y_force     ok
5      shock_z_force     ok
6   bushing_f_x_momt     ok
7   bushing_f_y_momt     ok
8   bushing_f_z_momt     ok
9   bushing_r_x_momt     ok
10  bushing_r_y_momt     ok
11  bushing_r_z_momt     ok
